# Chapter 58: Online Learning and Concept Drift Adaptation

## Learning Objectives
- Run prequential online learning
- Detect concept drift from rolling errors
- Adapt model updates after drift
- Compare static vs adaptive online performance

## Prerequisites
- Chapters 0-55 completed
- Strong understanding of ML systems and evaluation
- Optional matplotlib for richer visual outputs


# Chapter 58: Online Learning and Concept Drift Adaptation

## Why this chapter matters
In production streams, data distribution and label relationships evolve. Online learners and drift detectors help models adapt continuously.

## Learning goals
1. Implement prequential online logistic updates.
2. Simulate concept drift in streaming data.
3. Detect drift with rolling error monitoring.
4. Adapt model after drift and compare recovery.

## Prequential protocol
For each incoming sample:
1. predict
2. log error
3. update model with true label

## Drift strategy in this chapter
- rolling window error gap detector
- when drift flagged: increase learning rate and partial reset

## Assignment
1. Compare static model vs adaptive online model.
2. Sweep detector thresholds.
3. Plot accuracy over time with drift markers.


In [ ]:
from __future__ import annotations

import math
import random
from typing import List, Tuple


def sigmoid(z: float) -> float:
    return 1.0 / (1.0 + math.exp(-max(-35, min(35, z))))


def generate_stream(n: int = 4000, drift_at: int = 2200, seed: int = 42):
    random.seed(seed)
    X, y = [], []
    for i in range(n):
        x1 = random.uniform(-1, 1)
        x2 = random.uniform(-1, 1)
        x3 = 1.0
        if i < drift_at:
            logit = 1.3 * x1 - 1.0 * x2
        else:
            logit = -1.1 * x1 + 1.4 * x2 + 0.2
        p = sigmoid(logit)
        yi = 1 if random.random() < p else 0
        X.append([x1, x2, x3])
        y.append(yi)
    return X, y


def online_predict(x: List[float], w: List[float]) -> float:
    return sigmoid(sum(wi * xi for wi, xi in zip(w, x)))


def run_online(
    X: List[List[float]],
    y: List[int],
    adaptive: bool,
    base_lr: float = 0.03,
    win: int = 120,
    gap_thresh: float = 0.08,
):
    w = [0.0, 0.0, 0.0]
    errors = []
    acc_curve = []
    drift_points = []

    for i, (xi, yi) in enumerate(zip(X, y), 1):
        p = online_predict(xi, w)
        pred = 1 if p >= 0.5 else 0
        err = 0 if pred == yi else 1
        errors.append(err)

        # drift detection: recent window vs previous window
        lr = base_lr
        if adaptive and len(errors) >= 2 * win:
            a = sum(errors[-win:]) / win
            b = sum(errors[-2 * win : -win]) / win
            if a - b > gap_thresh:
                drift_points.append(i)
                lr = base_lr * 3.0
                # mild reset to adapt faster
                w = [0.7 * v for v in w]

        # online logistic update
        grad = yi - p
        for j in range(len(w)):
            w[j] += lr * grad * xi[j]

        # smoothed accuracy
        k = min(200, len(errors))
        acc = 1.0 - (sum(errors[-k:]) / k)
        acc_curve.append(acc)

    return acc_curve, drift_points


def maybe_plot(acc_static, acc_adapt, drift_points):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("Final rolling acc static:", round(acc_static[-1], 4))
        print("Final rolling acc adapt :", round(acc_adapt[-1], 4))
        print("Drift points detected   :", drift_points[:10])
        return

    plt.figure(figsize=(9, 4))
    plt.plot(acc_static, label="static online")
    plt.plot(acc_adapt, label="adaptive online")
    for d in drift_points[:10]:
        plt.axvline(d, color="red", alpha=0.15)
    plt.title("Prequential Accuracy Under Concept Drift")
    plt.xlabel("Time step")
    plt.ylabel("Rolling accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    X, y = generate_stream(n=4200, drift_at=2300, seed=11)
    acc_static, _ = run_online(X, y, adaptive=False)
    acc_adapt, drifts = run_online(X, y, adaptive=True, base_lr=0.03, win=120, gap_thresh=0.07)

    print("Final static acc:", round(acc_static[-1], 4))
    print("Final adapt  acc:", round(acc_adapt[-1], 4))
    print("Detected drift points:", drifts[:8], "... total", len(drifts))

    maybe_plot(acc_static, acc_adapt, drifts)


## Checkpoint

1. You can explain prequential evaluation protocol.
2. You can detect drift from error dynamics.
3. You can design adaptation strategy for recovery.

## Guided Exercise
Sweep drift threshold and observe false alarms vs adaptation speed.

In [ ]:
# TODO (Student Exercise)
# Test gap_thresh in [0.04, 0.07, 0.10] and compare final accuracy.

## Quick Quiz

**Q1.** Why can offline-trained models fail online?

**Answer:** Underlying concept changes over time.

**Q2.** What is prequential accuracy?

**Answer:** Streaming accuracy measured before each update.

**Q3.** Why not reset model fully on every alarm?

**Answer:** May lose useful historical structure and create instability.


## Homework
Implement a drift-aware ensemble with weighted old/new models.